In [2]:
# ============================================================
# TASK 1 - Data Ingestion Pipeline & Exploratory Analysis
# RenewCred EV Telemetry Intelligence
# ============================================================

import json
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import folium
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

SECRET_SALT = os.getenv("SECRET_SALT")
print("✅ Salt loaded successfully")

✅ Salt loaded successfully


In [10]:
# ============================================================
# SECTION 1.1 - Parse & Flatten the MQTT Payload
# ============================================================

import hashlib


def anonymise_imei(imei: str) -> str:
    """Hash IMEI using SHA-256 with salt - never store raw IMEI"""
    if pd.isna(imei) or imei == "":
        return None
    salted = f"{imei}{SECRET_SALT}".encode('utf-8')
    return hashlib.sha256(salted).hexdigest()

def parse_ev_payload(raw_csv_path: str) -> pd.DataFrame:
    """
    Parse raw MQTT CSV → clean flat DataFrame.
    Handles: malformed JSON, duplicate timestamps,
    missing battery/GPS nested objects, epoch-ms → UTC conversion.
    Returns DataFrame with all 21 columns from field reference.
    """
    
    # Load raw CSV
    print(f"📂 Loading file: {raw_csv_path}")
    raw_df = pd.read_csv(raw_csv_path)
    print(f"✅ Raw shape: {raw_df.shape}")
    print(f"📋 Raw columns: {list(raw_df.columns)}")
    
    return raw_df

# Run it
RAW_CSV_PATH = "../data/raw/EV _ Sample data - data-1773851228982 - EV.csv"
raw_df = parse_ev_payload(RAW_CSV_PATH)
raw_df.head()

📂 Loading file: ../data/raw/EV _ Sample data - data-1773851228982 - EV.csv
✅ Raw shape: (21409, 6)
📋 Raw columns: ['topic', 'payload', 'created_timestamp', 'Unnamed: 3', 'topic.1', 'payload.1']


,topic,payload,created_timestamp,Unnamed: 3,topic.1,payload.1
0,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0425A0600006"",...",46:31.2,NaN,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0425A0600006"",..."
1,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0425A0600006"",...",46:31.2,NaN,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0425A0600006"",..."
2,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",...",46:31.2,NaN,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",..."
3,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",...",46:31.2,NaN,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",..."
4,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",...",46:31.2,NaN,ev.3w.telemetry,"{\n ""payload"": {""deviceId"":""TEC0325A0600002"",..."


In [11]:
# ============================================================
# Inspect raw data before parsing
# ============================================================

# Check for duplicate columns
print("🔍 First few rows of payload column:")
print(raw_df['payload'].head(3))

print("\n🔍 First few rows of payload.1 column:")
print(raw_df['payload.1'].head(3))

print("\n🔍 Unnamed column sample:")
print(raw_df['Unnamed: 3'].head(3))

print("\n🔍 Null counts per column:")
print(raw_df.isnull().sum())

🔍 First few rows of payload column:
0    {\n  "payload": {"deviceId":"TEC0425A0600006",...
1    {\n  "payload": {"deviceId":"TEC0425A0600006",...
2    {\n  "payload": {"deviceId":"TEC0325A0600002",...
Name: payload, dtype: object

🔍 First few rows of payload.1 column:
0    {\n  "payload": {"deviceId":"TEC0425A0600006",...
1    {\n  "payload": {"deviceId":"TEC0425A0600006",...
2    {\n  "payload": {"deviceId":"TEC0325A0600002",...
Name: payload.1, dtype: object

🔍 Unnamed column sample:
0   NaN
1   NaN
2   NaN
Name: Unnamed: 3, dtype: float64

🔍 Null counts per column:
topic                    0
payload                  0
created_timestamp        0
Unnamed: 3           21409
topic.1                  0
payload.1                0
dtype: int64


In [12]:
# ============================================================
# SECTION 1.1 - Full Payload Parser
# ============================================================

from dotenv import load_dotenv
import os

load_dotenv()
SECRET_SALT = os.getenv("SECRET_SALT")

def anonymise_imei(imei):
    """Hash IMEI using SHA-256 with salt"""
    if pd.isna(imei) or str(imei).strip() == "":
        return None
    salted = f"{str(imei)}{SECRET_SALT}".encode('utf-8')
    return hashlib.sha256(salted).hexdigest()

def parse_single_payload(raw_json_str):
    """
    Parse a single JSON string from payload column.
    Returns a flat dict with all 21 fields.
    Returns None if malformed.
    """
    try:
        # Parse outer JSON
        outer = json.loads(raw_json_str)
        
        # Get inner payload
        p = outer.get('payload', {})
        
        # Get nested objects safely
        gps = p.get('gps', {}) or {}
        battery = p.get('battery', {}) or {}
        
        return {
            # Device info
            'device_id':            p.get('deviceId'),
            'imei_token':           anonymise_imei(p.get('imei')),
            'last_ping_time':       p.get('lastPingTime'),
            'device_status':        p.get('status'),
            
            # GPS fields
            'gps_lat':              gps.get('gpsLatitude'),
            'gps_lon':              gps.get('gpsLongitude'),
            'gps_speed_kmh':        gps.get('gpsGroundSpeed'),
            'gps_delta_km':         gps.get('gpsGroundDeltaDistance'),
            'gps_total_km':         gps.get('gpsTotalGroundDistance'),
            
            # Battery fields
            'battery_state':        battery.get('batteryState'),
            'battery_soc_pct':      battery.get('batterySoc'),
            'battery_capacity_ah':  battery.get('batteryInstalledCapacity'),
            'battery_usable_ah':    battery.get('batteryUsableCapacity'),
            'capacity_discharge_ah':battery.get('batteryCapacityToDischarge'),
            'capacity_charge_ah':   battery.get('batteryCapacityToCharge'),
            'battery_voltage_v':    battery.get('batteryVoltage'),
            'cell_voltage_min':     battery.get('batteryMinCellVoltage'),
            'cell_voltage_max':     battery.get('batteryMaxCellVoltage'),
            'battery_temp_c':       battery.get('batteryAvgTemp'),
            'battery_soh_pct':      battery.get('batterySoh'),
            
            # Timestamp
            'ts':                   outer.get('timestamp'),
        }
    except Exception as e:
        # Malformed row — skip it
        return None

def parse_ev_payload(raw_csv_path: str) -> pd.DataFrame:
    """
    Full pipeline: raw CSV → clean flat DataFrame
    """
    # Load raw CSV
    print(f"📂 Loading: {raw_csv_path}")
    raw_df = pd.read_csv(raw_csv_path)
    print(f"✅ Raw shape: {raw_df.shape}")
    
    # Drop useless columns
    raw_df = raw_df.drop(columns=['Unnamed: 3', 'topic.1', 'payload.1'], errors='ignore')
    print(f"✅ Dropped duplicate/empty columns")
    
    # Parse each payload row
    print(f"⏳ Parsing {len(raw_df)} rows...")
    parsed_rows = []
    malformed_count = 0
    
    for idx, row in raw_df.iterrows():
        result = parse_single_payload(row['payload'])
        if result is not None:
            parsed_rows.append(result)
        else:
            malformed_count += 1
    
    print(f"✅ Successfully parsed: {len(parsed_rows)} rows")
    print(f"⚠️  Malformed/skipped rows: {malformed_count}")
    
    # Build DataFrame
    df = pd.DataFrame(parsed_rows)
    
    # Convert timestamp epoch-ms → UTC datetime
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    
    # Sort by device and time
    df = df.sort_values(['device_id', 'ts']).reset_index(drop=True)
    
    # Remove exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"✅ Removed {before - after} exact duplicate rows")
    
    print(f"\n✅ Final DataFrame shape: {df.shape}")
    print(f"📱 Unique devices: {df['device_id'].nunique()}")
    print(f"📅 Date range: {df['ts'].min()} → {df['ts'].max()}")
    
    return df

# Run the parser
df = parse_ev_payload(RAW_CSV_PATH)
df.head()

📂 Loading: ../data/raw/EV _ Sample data - data-1773851228982 - EV.csv
✅ Raw shape: (21409, 6)
✅ Dropped duplicate/empty columns
⏳ Parsing 21409 rows...
✅ Successfully parsed: 21409 rows
⚠️  Malformed/skipped rows: 0
✅ Removed 0 exact duplicate rows

✅ Final DataFrame shape: (21409, 21)
📱 Unique devices: 3
📅 Date range: 2026-02-03 19:03:02.112000+00:00 → 2026-02-20 16:56:29.198000+00:00


,device_id,imei_token,last_ping_time,device_status,gps_lat,gps_lon,gps_speed_kmh,gps_delta_km,gps_total_km,battery_state,battery_soc_pct,battery_capacity_ah,battery_usable_ah,capacity_discharge_ah,capacity_charge_ah,battery_voltage_v,cell_voltage_min,cell_voltage_max,battery_temp_c,battery_soh_pct,ts
0,TEC0325A0600002,f1080e5a29468c27add4bcf24acbad03e05660994d7b0f...,2026-02-03T18:46:46Z,Inactive,11.0456,76.9005,1.0400,0.0000,820.0100,Idle,99,125,124.7600,123.5100,1.2500,7.9200,3.3300,3.3300,28.5100,99.8100,2026-02-03 19:04:02.116000+00:00
1,TEC0325A0600002,f1080e5a29468c27add4bcf24acbad03e05660994d7b0f...,2026-02-03T18:46:46Z,Inactive,11.0456,76.9005,1.0400,0.0000,820.0100,Idle,99,125,124.7600,123.5100,1.2500,7.9200,3.3300,3.3300,28.5100,99.8100,2026-02-03 19:04:32.116000+00:00
2,TEC0325A0600002,f1080e5a29468c27add4bcf24acbad03e05660994d7b0f...,2026-02-03T18:46:46Z,Inactive,11.0456,76.9005,1.0400,0.0000,820.0100,Idle,99,125,124.7600,123.5100,1.2500,7.9200,3.3300,3.3300,28.5100,99.8100,2026-02-03 19:05:02.111000+00:00
3,TEC0325A0600002,f1080e5a29468c27add4bcf24acbad03e05660994d7b0f...,2026-02-03T18:46:46Z,Inactive,11.0456,76.9005,1.0400,0.0000,820.0100,Idle,99,125,124.7600,123.5100,1.2500,7.9200,3.3300,3.3300,28.5100,99.8100,2026-02-03 19:06:02.122000+00:00
4,TEC0325A0600002,f1080e5a29468c27add4bcf24acbad03e05660994d7b0f...,2026-02-03T18:46:46Z,Inactive,11.0456,76.9005,1.0400,0.0000,820.0100,Idle,99,125,124.7600,123.5100,1.2500,7.9200,3.3300,3.3300,28.5100,99.8100,2026-02-03 19:07:02.120000+00:00


In [13]:
# ============================================================
# Sanity check - verify all 21 columns and data types
# ============================================================

print("📋 Column names and data types:")
print(df.dtypes)

print(f"\n📊 Data per device:")
print(df.groupby('device_id').agg(
    total_rows=('ts', 'count'),
    date_from=('ts', 'min'),
    date_to=('ts', 'max')
).to_string())

print(f"\n🔋 Battery state distribution:")
print(df['battery_state'].value_counts())

print(f"\n📍 Sample GPS coordinates:")
print(df[['device_id', 'gps_lat', 'gps_lon']].dropna().head(5))

📋 Column names and data types:
device_id                             object
imei_token                            object
last_ping_time                        object
device_status                         object
gps_lat                              float64
gps_lon                              float64
gps_speed_kmh                        float64
gps_delta_km                         float64
gps_total_km                         float64
battery_state                         object
battery_soc_pct                        int64
battery_capacity_ah                    int64
battery_usable_ah                    float64
capacity_discharge_ah                float64
capacity_charge_ah                   float64
battery_voltage_v                    float64
cell_voltage_min                     float64
cell_voltage_max                     float64
battery_temp_c                       float64
battery_soh_pct                      float64
ts                       datetime64[ns, UTC]
dtype: object

📊 Data pe

In [14]:
# ============================================================
# Fix data types
# ============================================================

# Convert to float where needed
df['battery_capacity_ah'] = df['battery_capacity_ah'].astype(float)
df['battery_soc_pct'] = df['battery_soc_pct'].astype(float)

# Convert last_ping_time to datetime
df['last_ping_time'] = pd.to_datetime(df['last_ping_time'], utc=True, errors='coerce')

# Convert categorical columns
df['battery_state'] = df['battery_state'].astype('category')
df['device_status'] = df['device_status'].astype('category')

print("✅ Data types fixed")
print(f"\n📋 Updated dtypes:")
print(df[['battery_capacity_ah', 'battery_soc_pct', 
          'last_ping_time', 'battery_state', 
          'device_status']].dtypes)

print(f"\n📊 Device status distribution:")
print(df['device_status'].value_counts())

print(f"\n🔋 Battery SoC range:")
print(f"Min: {df['battery_soc_pct'].min()}%")
print(f"Max: {df['battery_soc_pct'].max()}%")
print(f"Mean: {df['battery_soc_pct'].mean():.2f}%")

✅ Data types fixed

📋 Updated dtypes:
battery_capacity_ah                float64
battery_soc_pct                    float64
last_ping_time         datetime64[ns, UTC]
battery_state                     category
device_status                     category
dtype: object

📊 Device status distribution:
device_status
Inactive    19319
Active       2090
Name: count, dtype: int64

🔋 Battery SoC range:
Min: 0.0%
Max: 99.0%
Mean: 47.37%


In [15]:
# ============================================================
# SECTION 1.2 - Data Quality Assessment
# ============================================================

print("=" * 60)
print("DATA QUALITY REPORT - RenewCred EV Telemetry")
print("=" * 60)

# ── 1. Null rates per column ──────────────────────────────
print("\n📊 1. NULL RATES PER COLUMN")
print("-" * 40)

null_rates = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_rate_%': null_rates
})

# Signal columns that matter most
signal_cols = ['battery_soc_pct', 'battery_voltage_v', 
               'gps_lat', 'gps_lon', 'cell_voltage_min', 
               'cell_voltage_max', 'battery_temp_c', 'battery_soh_pct']

print(null_df.to_string())

print("\n⚠️  Signal columns with > 5% nulls:")
flagged = null_df[null_df.index.isin(signal_cols) & (null_df['null_rate_%'] > 5)]
if len(flagged) == 0:
    print("✅ No signal columns exceed 5% null threshold")
else:
    print(flagged.to_string())

DATA QUALITY REPORT - RenewCred EV Telemetry

📊 1. NULL RATES PER COLUMN
----------------------------------------
                       null_count  null_rate_%
device_id                       0       0.0000
imei_token                      0       0.0000
last_ping_time                  0       0.0000
device_status                   0       0.0000
gps_lat                         0       0.0000
gps_lon                         0       0.0000
gps_speed_kmh                   0       0.0000
gps_delta_km                    0       0.0000
gps_total_km                    0       0.0000
battery_state                   0       0.0000
battery_soc_pct                 0       0.0000
battery_capacity_ah             0       0.0000
battery_usable_ah               0       0.0000
capacity_discharge_ah           0       0.0000
capacity_charge_ah              0       0.0000
battery_voltage_v               0       0.0000
cell_voltage_min                0       0.0000
cell_voltage_max                0       

In [16]:
# ── 2. Duplicate Detection ────────────────────────────────
print("\n📊 2. DUPLICATE DETECTION")
print("-" * 40)

# Exact duplicate rows
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

# Same device_id + ts within 1-second window
print("\n⏱️  Checking same device + timestamp within 1-second window...")
df_sorted = df.sort_values(['device_id', 'ts'])

# Round timestamps to nearest second for comparison
df_sorted['ts_rounded'] = df_sorted['ts'].dt.round('1s')

near_dupes = df_sorted.duplicated(
    subset=['device_id', 'ts_rounded']
).sum()

print(f"Near-duplicate rows (same device + ts within 1 sec): {near_dupes}")

# ── 3. Out of Range Flags ─────────────────────────────────
print("\n📊 3. OUT OF RANGE FLAGS")
print("-" * 40)

# SoC outside [0, 100]
soc_oor = df[(df['battery_soc_pct'] < 0) | (df['battery_soc_pct'] > 100)]
print(f"battery_soc_pct outside [0,100]:    {len(soc_oor)} rows")

# SoH outside [0, 100]
soh_oor = df[(df['battery_soh_pct'] < 0) | (df['battery_soh_pct'] > 100)]
print(f"battery_soh_pct outside [0,100]:    {len(soh_oor)} rows")

# Cell voltage outside [2.5V, 4.2V]
volt_oor = df[
    (df['cell_voltage_min'] < 2.5) | 
    (df['cell_voltage_max'] > 4.2)
]
print(f"cell_voltage outside [2.5V, 4.2V]:  {len(volt_oor)} rows")

if len(volt_oor) > 0:
    print("\n⚠️  Out of range voltage samples:")
    print(volt_oor[['device_id', 'ts', 'cell_voltage_min', 
                     'cell_voltage_max']].head(5).to_string())


📊 2. DUPLICATE DETECTION
----------------------------------------
Exact duplicate rows: 0

⏱️  Checking same device + timestamp within 1-second window...
Near-duplicate rows (same device + ts within 1 sec): 0

📊 3. OUT OF RANGE FLAGS
----------------------------------------
battery_soc_pct outside [0,100]:    0 rows
battery_soh_pct outside [0,100]:    0 rows
cell_voltage outside [2.5V, 4.2V]:  0 rows


In [17]:
# ── 4. Temporal Continuity ────────────────────────────────
print("\n📊 4. TEMPORAL CONTINUITY - INTER-PING GAP ANALYSIS")
print("-" * 40)

# Calculate time gap between consecutive pings per device
df_sorted = df.sort_values(['device_id', 'ts'])
df_sorted['ping_gap_sec'] = (
    df_sorted.groupby('device_id')['ts']
    .diff()
    .dt.total_seconds()
)

# Summary per device
print("📱 Inter-ping gap statistics per device (seconds):")
gap_stats = df_sorted.groupby('device_id')['ping_gap_sec'].agg([
    'count', 'mean', 'median', 'min', 'max'
]).round(2)
print(gap_stats.to_string())

# Flag devices with > 10% of pings delayed > 60 sec
print("\n⚠️  Devices with > 10% pings delayed > 60 seconds:")
flagged_devices = []

for device, group in df_sorted.groupby('device_id'):
    gaps = group['ping_gap_sec'].dropna()
    total_pings = len(gaps)
    delayed_pings = (gaps > 60).sum()
    delayed_pct = (delayed_pings / total_pings * 100).round(2)
    
    status = "⚠️  FLAGGED" if delayed_pct > 10 else "✅ OK"
    print(f"{status} | {device} | "
          f"delayed pings: {delayed_pings}/{total_pings} "
          f"({delayed_pct}%)")
    
    if delayed_pct > 10:
        flagged_devices.append(device)

print(f"\nTotal flagged devices: {len(flagged_devices)}")

# Add ping_gap_sec to main df
df['ping_gap_sec'] = df_sorted['ping_gap_sec']


📊 4. TEMPORAL CONTINUITY - INTER-PING GAP ANALYSIS
----------------------------------------
📱 Inter-ping gap statistics per device (seconds):
                 count     mean  median     min          max
device_id                                                   
TEC0325A0600002   7628 191.5300 30.0000 29.1000 1229422.9200
TEC0425A0600006   7628 191.5300 30.0000 29.1000 1229362.9200
TEC0725A0600013   6150 230.2600 30.0000 29.1000 1229452.9200

⚠️  Devices with > 10% pings delayed > 60 seconds:
✅ OK | TEC0325A0600002 | delayed pings: 32/7628 (0.42%)
✅ OK | TEC0425A0600006 | delayed pings: 32/7628 (0.42%)
✅ OK | TEC0725A0600013 | delayed pings: 25/6150 (0.41%)

Total flagged devices: 0


In [18]:
# ── 5. Cell Voltage Imbalance ─────────────────────────────
print("\n📊 5. CELL VOLTAGE IMBALANCE ANALYSIS")
print("-" * 40)

# Calculate imbalance
df['cell_imbalance'] = df['cell_voltage_max'] - df['cell_voltage_min']

# Summary stats
print("📊 Cell imbalance statistics (Volts):")
print(df.groupby('device_id')['cell_imbalance'].agg([
    'mean', 'median', 'max', 'std'
]).round(4).to_string())

# Flag dangerous imbalance > 0.05V
dangerous = df[df['cell_imbalance'] > 0.05]
print(f"\n⚠️  Readings with cell imbalance > 0.05V: {len(dangerous)}")

if len(dangerous) > 0:
    print("\nSample dangerous readings:")
    print(dangerous[['device_id', 'ts', 'cell_voltage_min',
                      'cell_voltage_max', 
                      'cell_imbalance']].head(5).to_string())

# Overall quality summary
print("\n" + "=" * 60)
print("✅ DATA QUALITY SUMMARY")
print("=" * 60)
print(f"Total rows:              {len(df):,}")
print(f"Null values:             0")
print(f"Exact duplicates:        0")
print(f"Near duplicates:         0")
print(f"Out of range SoC:        0")
print(f"Out of range SoH:        0")
print(f"Out of range voltage:    0")
print(f"Flagged devices:         0")
print(f"Dangerous imbalance:     {len(dangerous):,}")
print(f"Unique devices:          {df['device_id'].nunique()}")
print(f"Date range:              Feb 03 → Feb 20, 2026")
print("=" * 60)


📊 5. CELL VOLTAGE IMBALANCE ANALYSIS
----------------------------------------
📊 Cell imbalance statistics (Volts):
                  mean  median    max    std
device_id                                   
TEC0325A0600002 0.0418  0.0400 0.1900 0.0432
TEC0425A0600006 0.0207  0.0200 0.0300 0.0025
TEC0725A0600013 0.0329  0.0100 0.1500 0.0372

⚠️  Readings with cell imbalance > 0.05V: 3847

Sample dangerous readings:
            device_id                               ts  cell_voltage_min  cell_voltage_max  cell_imbalance
1171  TEC0325A0600002 2026-02-04 04:58:32.109000+00:00            3.4000            3.5400          0.1400
1172  TEC0325A0600002 2026-02-04 04:59:02.109000+00:00            3.4000            3.5400          0.1400
1173  TEC0325A0600002 2026-02-04 04:59:32.114000+00:00            3.4000            3.5400          0.1400
1174  TEC0325A0600002 2026-02-04 05:00:02.167000+00:00            3.4000            3.5400          0.1400
1175  TEC0325A0600002 2026-02-04 05:00:32.231000

In [19]:
# ============================================================
# Save cleaned DataFrame for use in Tasks 2 & 3
# ============================================================

# Save to outputs folder
df.to_csv('../outputs/ev_telemetry_clean.csv', index=False)
print("✅ Clean DataFrame saved to outputs/ev_telemetry_clean.csv")

# Quick final check
print(f"\n📋 Final DataFrame shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

✅ Clean DataFrame saved to outputs/ev_telemetry_clean.csv

📋 Final DataFrame shape: (21409, 23)
📋 Columns: ['device_id', 'imei_token', 'last_ping_time', 'device_status', 'gps_lat', 'gps_lon', 'gps_speed_kmh', 'gps_delta_km', 'gps_total_km', 'battery_state', 'battery_soc_pct', 'battery_capacity_ah', 'battery_usable_ah', 'capacity_discharge_ah', 'capacity_charge_ah', 'battery_voltage_v', 'cell_voltage_min', 'cell_voltage_max', 'battery_temp_c', 'battery_soh_pct', 'ts', 'ping_gap_sec', 'cell_imbalance']
